<a href="https://colab.research.google.com/github/Yike-Ding/structural_bioinformatics/blob/main/02_binding_site_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**02 Binding Site Identification**

The histone family is a key component of nucleosome. It plays a central role in transcription regulation, DNA replication and chromosomal stability.

With the isolated histone 3.1 structure obtained, we examine whether it is a suitable target by evaluating potential binding sites.

In [4]:
import numpy
import matplotlib.pyplot
import os
import requests
import urllib.request


In [6]:
GITHUB_RAW_PDB_URL = "https://raw.githubusercontent.com/Yike-Ding/structural_bioinformatics/main/data/3AFA_H3.1.pdb"
pdb_filename = "3AFA_H3.1.pdb"

if not os.path.exists(pdb_filename):
  print(f"Downloading {pdb_filename} from GitHub...")
  urllib.request.urlretrieve(GITHUB_RAW_PDB_URL, pdb_filename)
  print("Download completed!")
else:
  print(f"{pdb_filename} already exists.")

Download completed!


Assign a pdb code from our customized pdb file at the protein plus site.

In [46]:
import time

def pdb_file_to_pdb_code(pdb_path: str,max_retries: int = 20, delay: int = 2) -> str:
    """Upload a local PDB file to ProteinsPlus and retrieve a temporary PDB code

    (ID) that can be passed to subsequent tools (like DoGSite3).

    Parameters
    ----------
    pdb_path : str
        Path to the local processed PDB file (e.g., 'protein_processed.pdb').

    Returns
    -------
    str
        Temporary PDB identifier assigned by ProteinsPlus.
    """
    if not os.path.exists(pdb_path):
      raise FileNotFoundError(f"PDB file not found: {pdb_path}")

    url = "https://proteins.plus/api/pdb_files_rest"
    headers = {"Accept": "application/json"}

    with open(pdb_path, "rb") as f:
      files = {"pdb_file[pathvar]": (os.path.basename(pdb_path), f)}
      response = requests.post(url, files=files, headers=headers)

    response.raise_for_status()
    res_data = response.json()


    if "id" in res_data:
      return res_data["id"]

    if "location" in res_data:
      poll_url = res_data["location"]

      custom_id = poll_url.rstrip("/").split("/")[-1]

      for _ in range(max_retries):
        poll_resp = requests.get(poll_url, headers=headers)
        poll_resp.raise_for_status()
        status_data = poll_resp.json()

        if status_data.get("status_code") == 200 or "id" in status_data:
          return status_data.get("id", custom_id)

        time.sleep(delay)

      return custom_id

    raise KeyError(f"Unexpected response structure: {res_data}")

In [47]:
def submit_dogsite3_job_with_pdbid(pdb_code: str, chain_id: str, ligand: str = ""):
  """
  Submit pdb code to DoGSiteScorer API and get back url.

  Parameters
  ---------
  pdb file

  Returns
  ---------
  str
        The job location URL (relative ID) to poll for results.
  """
  url = "https://proteins.plus/api/dogsite3_rest"
  payload = {
      "dogsite3" :{
        "pdbCode":pdb_code,
        "analysisDetail" : "1",
        "bindingSitePredictionGranularity": "1",
        "ligand": ligand,
        "chain":chain_id,
        "ligandBias": "0"
    }
  }
  headers = {"Accept": "application/json", "Content-Type": "application/json"}

  r = requests.post(url, json=payload, headers=headers)
  r.raise_for_status()
  resp = r.json()

  # Should return status_code + location (job ID)
  if "location" in resp:
      return resp["location"]
  else:
      raise RuntimeError("Unexpected response from DoGSite3 submission: %r" % resp)


In [48]:
assigned_code = pdb_file_to_pdb_code("3AFA_H3.1.pdb")
print("Assigned PDB Code:", assigned_code)

job_url = submit_dogsite3_job_with_pdbid(assigned_code, chain_id="A")
print("Job URL:", job_url)

Assigned PDB Code: 3afah31pdbb9f9d8bb-8aa0-4640-8dc1-7a3b201d863b
Job URL: https://proteins.plus/api/dogsite3_rest/626N5mvuiYPEhWf8QGAtQikm


In [49]:
import pandas as pd
import io

def get_dogsitescorer_metadata(job_url, attempts=30):
    """
    Get results from a DoGSiteScorer query, i.e., the binding sites which are found over the protein surface,
    in the form of a table with the details about all detected pockets.

    Parameters
    ----------
    job_location : str
        Consists of the location of a finished DoGSiteScorer job on the proteins.plus web server.
    attempts : int
        The time waiting for the feedback from DoGSiteScorer service.

    Returns
    -------
    pandas.DataFrame
        Table with metadata on detected binding sites.
    """

    print(f"Querying for job at URL {job_url}...", end="")

    while attempts:
        # Get job results
        result = requests.get(job_url)
        result.raise_for_status()
        # Get URL of result table file
        response = result.json()
        if "result_table" in response:
            result_file = response["result_table"]
            break
        attempts -= 1
        print(".", end="")
        time.sleep(10)
    # Get result table (as string)
    result_table = requests.get(result_file).text
    # Load the table (csv format using "\t" as separator) with pandas DataFrame
    # We cannot load the table from a string directly but from a file
    # Use io.StringIO to wrap this string as file-like object as needed for read_csv method
    # See more: https://docs.python.org/3/library/io.html#io.StringIO
    result_table_df = pd.read_csv(io.StringIO(result_table), sep="\t").set_index("name")

    return result_table_df

In [50]:
metadata = get_dogsitescorer_metadata(job_url)
metadata.head()

Querying for job at URL https://proteins.plus/api/dogsite3_rest/626N5mvuiYPEhWf8QGAtQikm....

,lig_cov,poc_cov,lig_name,4A_crit,ligSASRatio,volume,enclosure,surface,lipoSurface,depth,...,G,U,I,N,DA,DC,DG,DT,DN,UNK
name,,,,,,,,,,,,,,,,,,,,,
P_1,0,0,NaN,0,0,107.008,0.790698,359.284,236.256,7.37564,...,0,0,0,0,0,0,0,0,0,0
P_2,0,0,NaN,0,0,79.360,0.734266,346.047,283.562,8.35225,...,0,0,0,0,0,0,0,0,0,0
P_2_1,0,0,NaN,0,0,46.080,0.687500,281.862,271.008,5.12250,...,0,0,0,0,0,0,0,0,0,0
P_2_2,0,0,NaN,0,0,33.280,0.793651,206.593,153.639,4.86621,...,0,0,0,0,0,0,0,0,0,0
P_3,0,0,NaN,0,0,64.512,0.836364,264.373,161.192,5.65685,...,0,0,0,0,0,0,0,0,0,0


In [51]:
cols_to_show = [
    c
    for c in ["volume", "surface", "lipoSurface", "depth", "dscore"]
    if c in metadata.columns
]
metadata[cols_to_show]

,volume,surface,lipoSurface,depth
name,,,,
P_1,107.008,359.284,236.256,7.37564
P_2,79.360,346.047,283.562,8.35225
P_2_1,46.080,281.862,271.008,5.12250
P_2_2,33.280,206.593,153.639,4.86621
P_3,64.512,264.373,161.192,5.65685
